# Student datasets

The four scenes from `generate_scenes.py`. Run that script first.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

SCENE_DIR = Path.cwd() / "scenes"
SCENES = ["flat", "flat_building", "irregular", "irregular_building"]

config = json.loads((SCENE_DIR / "config.json").read_text())
for k, v in config.items():
    print(f"{k:20s} {v}")

## Load

In [ ]:
def read_ply(path):
    with open(path, "rb") as f:
        n_v = n_f = 0
        while True:
            line = f.readline().decode("ascii").strip()
            if line.startswith("element vertex"):
                n_v = int(line.split()[-1])
            elif line.startswith("element face"):
                n_f = int(line.split()[-1])
            elif line == "end_header":
                break
        v = np.frombuffer(f.read(12 * n_v), dtype="<f4").reshape(n_v, 3)
        rec = np.frombuffer(f.read(13 * n_f),
                            dtype=[("n", "u1"), ("idx", "<i4", (3,))])
    return v, rec["idx"]

data = {}
for name in SCENES:
    d = SCENE_DIR / name
    data[name] = {
        "sources": dict(np.load(d / "sources.npz")),
        "height": dict(np.load(d / "height_map.npz")),
        "meshes": {p.stem: read_ply(p) for p in sorted((d / "meshes").glob("*.ply"))},
    }
    m = data[name]["meshes"]
    print(f"{name:20s} sources {len(data[name]['sources']['ids']):5d}   "
          + "   ".join(f"{k}: {len(f)} triangles" for k, (v, f) in m.items()))

## Top views
Ground height with the building in grey and the sources, coloured by height above ground.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# same colour scale in every panel, building drawn on top in grey
zmin = min(data[n]["height"]["terrain"].min() for n in SCENES)
zmax = max(data[n]["height"]["terrain"].max() for n in SCENES)
hmin = min(data[n]["sources"]["height_agl"].min() for n in SCENES)
hmax = max(data[n]["sources"]["height_agl"].max() for n in SCENES)

fig, axes = plt.subplots(2, 2, figsize=(11, 11), constrained_layout=True)
for ax, name in zip(axes.ravel(), SCENES):
    h, s = data[name]["height"], data[name]["sources"]
    extent = [h["x"][0], h["x"][-1], h["y"][0], h["y"][-1]]
    im = ax.imshow(h["terrain"], origin="lower", cmap="terrain",
                   vmin=zmin, vmax=zmax, extent=extent)
    building = np.ma.masked_where(h["surface"] <= h["terrain"], h["surface"])
    ax.imshow(building, origin="lower", cmap=ListedColormap(["dimgray"]),
              extent=extent)
    sc = ax.scatter(s["positions"][:, 0], s["positions"][:, 1], c=s["height_agl"],
                    s=4, cmap="plasma", vmin=hmin, vmax=hmax)
    ax.set_title(name)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
fig.colorbar(im, ax=axes, location="right", shrink=0.6, label="ground height [m]")
fig.colorbar(sc, ax=axes, location="bottom", shrink=0.6, aspect=40,
             label="source height above ground [m]")
fig.legend(handles=[Patch(color="dimgray",
                          label=f"building ({config['BUILDING_HEIGHT']:.0f} m)")],
           loc="outside upper center")
plt.show()

## Profiles
Cut along y = 0, through the building.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for name in SCENES:
    h = data[name]["height"]
    iy = np.argmin(np.abs(h["y"]))
    ax.plot(h["x"], h["surface"][iy], label=name)
ax.set_xlabel("x [m]")
ax.set_ylabel("height [m]")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Sources
Same sources in every scene, only z follows the terrain. For a Poisson process the nearest neighbour distance is $p(r) = 2\pi\lambda r\,e^{-\lambda\pi r^2}$.

In [ ]:
s_flat = data["flat"]["sources"]
s_irr = data["irregular"]["sources"]
xy = s_flat["positions"][:, :2]
lam = config["SOURCE_DENSITY"]

d = np.linalg.norm(xy[:, None, :] - xy[None, :, :], axis=-1)
np.fill_diagonal(d, np.inf)
nn = d.min(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(s_flat["height_agl"], bins=30, color="tab:blue")
axes[0].set_xlabel("height above ground [m]")
axes[0].set_ylabel("sources")

axes[1].hist(s_flat["positions"][:, 2], bins=30, alpha=0.6, label="flat")
axes[1].hist(s_irr["positions"][:, 2], bins=30, alpha=0.6, label="irregular")
axes[1].set_xlabel("absolute height z [m]")
axes[1].legend()

r = np.linspace(0, nn.max(), 200)
axes[2].hist(nn, bins=30, density=True, alpha=0.6, label="sources")
axes[2].plot(r, 2 * np.pi * lam * r * np.exp(-lam * np.pi * r ** 2), "k", label="Poisson theory")
axes[2].set_xlabel("nearest neighbour distance [m]")
axes[2].legend()
plt.tight_layout()
plt.show()

print(f"sources: {len(nn)}")
print(f"nearest neighbour distance: median {np.median(nn):.2f} m, "
      f"min {nn.min():.2f} m, max {nn.max():.2f} m")

## 3D view
Change `name` for another scene.

In [ ]:
def show_3d(name):
    fig = go.Figure()
    colors = {"ground": "darkkhaki", "building": "lightgray"}
    for key, (v, f) in data[name]["meshes"].items():
        fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2],
                                i=f[:, 0], j=f[:, 1], k=f[:, 2],
                                color=colors.get(key, "gray"), opacity=1.0,
                                name=key, flatshading=True))
    p = data[name]["sources"]["positions"]
    fig.add_trace(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
                               marker=dict(size=2, color="red"), name="sources"))
    fig.update_layout(title=name, height=650,
                      scene=dict(aspectmode="data", xaxis_title="x [m]",
                                 yaxis_title="y [m]", zaxis_title="z [m]"))
    fig.show()

name = "irregular_building"
show_3d(name)

## Sionna render

In [ ]:
from sionna.rt import load_scene, Camera

scene = load_scene(str(SCENE_DIR / name / f"{name}.xml"))
print("objects:", list(scene.objects.keys()))
cam = Camera(position=[-70, -70, 50], look_at=[0, 0, 0])
scene.render(camera=cam, resolution=(800, 500))
plt.show()